# Double Mass Curve & Flow Deviation Analysis

**Purpose:** Generates a four-panel diagnostic plot to assess the consistency
and bias between simulated and observed discharge over the study period.

**What it does:**
- Panel 1: Double mass curve (cumulative simulated vs. cumulative observed)
- Panel 2: Daily flow deviation (Sim − Obs)
- Panel 3: Cumulative precipitation (full record)
- Panel 4: Reservoir water level (full record)
- All panels share a time axis where applicable

**Input:** `Loop_6_TimeSeries_ZR.csv` (precip),
           `NT_Water_Level.xlsx` (reservoir level),
           `Deviation_and_Reservoir.xlsx` (discharge deviation)  
**Output:** Four-panel diagnostic figure

---

In [ ]:

"""
Multipanel hydrology plot:
- Panel 1: Double Mass (Sim cumulative vs Obs cumulative) -- clipped to START..END
- Panel 2: Daily Flow Deviation (Sim - Obs) -- clipped to START..END
- Panel 3: Cumulative Precipitation (FULL range, line only)
- Panel 4: Reservoir Water Level (FULL range, simple date vs WL line)

Author: Ahmed Rayhan
"""


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import matplotlib.ticker as mticker


In [ ]:
precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
res_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\NT_Water_Level.xlsx"
ard_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Deviation_and_Reservoir.xlsx"

In [ ]:
START = pd.Timestamp("2015-01-01")
END   = pd.Timestamp("2023-12-31")
SECONDS_PER_DAY = 86400

def build_reservoir_datetime(df_res: pd.DataFrame) -> pd.Series:
    df_res.columns = [c.strip() for c in df_res.columns]
    if "datetime" in df_res.columns:
        dt = pd.to_datetime(df_res["datetime"], dayfirst=True, errors="coerce")
    elif ("Datum" in df_res.columns) and ("Zeit" in df_res.columns):
        dt = pd.to_datetime(df_res["Datum"].astype(str).str.strip() + " " +
                            df_res["Zeit"].astype(str).str.strip(),
                            dayfirst=True, errors="coerce")
    elif "Datum" in df_res.columns:
        dt = pd.to_datetime(df_res["Datum"], dayfirst=True, errors="coerce")
    else:
        raise ValueError("Reservoir file must contain 'datetime' or 'Datum'.")
    return dt

# -----------------------------
# OBSERVED & SIMULATED FLOW (pre-processed daily volumes)
# -----------------------------
dev_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Deviation_and_Reservoir.xlsx"

df = pd.read_excel(dev_path, decimal=',')
df.columns = df.columns.str.strip()
df["Zeit"] = pd.to_datetime(df["Zeit"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Zeit"])
df = df[(df["Zeit"] >= START) & (df["Zeit"] <= END)].sort_values("Zeit").reset_index(drop=True)

# Detect and rename obs/sim daily volume columns
obs_col = [c for c in df.columns if "obs" in c.lower()][0]
sim_col = [c for c in df.columns if "sim" in c.lower()][0]
df.rename(columns={obs_col: "Obs_Daily_m3", sim_col: "Sim_Daily_m3"}, inplace=True)

# Back-calculate discharge (m3/s) and derive cumulative & deviation
df["Obs_Daily_m3s"]     = df["Obs_Daily_m3"] / SECONDS_PER_DAY
df["Sim_Daily_m3s"]     = df["Sim_Daily_m3"] / SECONDS_PER_DAY
df["Cum_Obs_m3"]        = df["Obs_Daily_m3"].cumsum()
df["Cum_Sim_m3"]        = df["Sim_Daily_m3"].cumsum()
df["Flow_Deviation_m3s"] = df["Sim_Daily_m3s"] - df["Obs_Daily_m3s"]

# -----------------------------
# PRECIPITATION: hourly -> daily sum (FULL range) + cumulative (line only)
# -----------------------------
df_precip = pd.read_csv(precip_path, sep=';')
df_precip.columns = df_precip.columns.str.strip()
if "datetime" not in df_precip.columns or "precip" not in df_precip.columns:
    raise ValueError("Precip file must have columns 'datetime' and 'precip'.")
df_precip["datetime"] = pd.to_datetime(df_precip["datetime"], dayfirst=True, errors="coerce")
df_precip = df_precip.dropna(subset=["datetime"])

df_precip_daily = (
    df_precip.set_index("datetime").resample("D")["precip"].sum().reset_index()
)
df_precip_daily = df_precip_daily[
    (df_precip_daily["datetime"] >= START) &
    (df_precip_daily["datetime"] <= END)
].reset_index(drop=True)

# Clean negatives (if any)
df_precip_daily["precip"] = df_precip_daily["precip"].clip(lower=0)
# Cumulative precipitation (mm) over the full range
df_precip_daily["precip_cum"] = df_precip_daily["precip"].cumsum()

# -----------------------------
# RESERVOIR: daily mean (FULL range)
# -----------------------------
df_res = pd.read_excel(res_path)
df_res.columns = df_res.columns.str.strip()

df_res["datetime"] = build_reservoir_datetime(df_res)

if "WL" not in df_res.columns:
    raise ValueError("Reservoir file must contain 'WL' column.")
df_res["WL"] = pd.to_numeric(
    df_res["WL"].astype(str).str.replace(",", ".", regex=False).str.strip(),
    errors="coerce"
)
df_res = df_res.dropna(subset=["datetime", "WL"]).sort_values("datetime")

# Daily aggregation: mean (switch to .last() if Excel uses last-of-day)
df_res_daily = (
    df_res.set_index("datetime").resample("D")["WL"].mean()
    .reset_index().rename(columns={"datetime": "Zeit"})
)
df_res_daily = df_res_daily[
    (df_res_daily["Zeit"] >= START) &
    (df_res_daily["Zeit"] <= END)
].reset_index(drop=True)

# -----------------------------
# CALCULATED RESERVOIR DATA
# -----------------------------
df_ard = pd.read_excel(ard_path, decimal=',')
df_ard["Zeit"] = pd.to_datetime(df_ard["Zeit"], dayfirst=True, errors="coerce")

# Rename the balance column to ARR_m3
balance_col = [c for c in df_ard.columns if "Balance" in c or "balance" in c][0]
df_ard.rename(columns={balance_col: "ARR_m3"}, inplace=True)

# Keep only relevant columns
df_ard = df_ard.dropna(subset=["Zeit", "ARR_m3"])
df_ard = df_ard[(df_ard["Zeit"] >= START) & (df_ard["Zeit"] <= END)].sort_values("Zeit")

In [ ]:
# Plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import numpy as np

PLOT_START = START
PLOT_END   = END

# ── Global font settings ──────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':   'monospace',
    'font.size':     8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 8,
})

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(6.5, 10))

gs = gridspec.GridSpec(
    nrows=5, ncols=1,
    figure=fig,
    height_ratios=[5.5, 2, 2, 2, 2],
    hspace=0.45                         # ← reduced from 0.55
)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2], sharex=ax1)
ax3 = fig.add_subplot(gs[3], sharex=ax1)
ax4 = fig.add_subplot(gs[4], sharex=ax1)

for ax in [ax1, ax2, ax3]:
    plt.setp(ax.get_xticklabels(), visible=False)
    ax.set_xlabel("")

# ── Helper ────────────────────────────────────────────────────────────────────
def fmt_time_axis(ax):
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[3, 6, 9, 12]))
    ax.grid(True, which="major", linestyle="--", alpha=0.5)

# ── Panel (a): Double-Mass ────────────────────────────────────────────────────
dm_max = max(df['Cum_Obs_m3'].max(), df['Cum_Sim_m3'].max())
ax0.plot([0, dm_max], [0, dm_max], 'k--', alpha=0.7, lw=1.2, label='1:1 line')
ax0.plot(df['Cum_Obs_m3'], df['Cum_Sim_m3'], color='blue', lw=1.5,
         label='Cumulative Sim vs Obs')

valid = df[['Cum_Obs_m3','Cum_Sim_m3','Obs_Daily_m3','Sim_Daily_m3']].dropna()
slope, _ = np.polyfit(valid['Cum_Obs_m3'], valid['Cum_Sim_m3'], 1)
pbias = (100 * (valid['Sim_Daily_m3'].sum() - valid['Obs_Daily_m3'].sum())
         / valid['Obs_Daily_m3'].sum())

ax0.text(0.04, 0.92, f"Slope = {slope:.3f}\nPBIAS = {pbias:.2f} %",
         transform=ax0.transAxes, fontsize=8, va='top',
         bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.3'))
ax0.set_xlabel("Cumulative Observed Volume (m³/d)")
ax0.set_ylabel("Cumulative Simulated Volume (m³/d)")
ax0.set_title("(a) Double Mass Curve", loc='left', fontweight='bold')
ax0.set_xlim(0, dm_max)
ax0.set_ylim(0, dm_max)
ax0.legend(loc='lower right')
ax0.grid(True, alpha=0.5)
ax0.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

# ── Panel (b): Flow Deviation ─────────────────────────────────────────────────
colors_flow = np.where(df['Flow_Deviation_m3s'] >= 0, '#1f77b4', '#d62728')
ax1.bar(df['Zeit'], df['Flow_Deviation_m3s'],
        color=colors_flow, width=np.timedelta64(1,'D'), align='center')
ax1.axhline(0, color='black', lw=0.8)
ax1.set_ylabel("Dev. (m³/s)")
ax1.set_title("(b) Daily Flow Deviation: Sim − Obs",
              loc='left', fontweight='bold', pad=6)
ax1.text(0.01, 0.95, "Blue = overestimation  |  Red = underestimation",
         transform=ax1.transAxes, fontsize=7,
         va='top', ha='left',          # ← top left
         bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))
fmt_time_axis(ax1)

# ── Panel (c): Cumulative Precipitation ───────────────────────────────────────
ax2.plot(df_precip_daily["datetime"], df_precip_daily["precip_cum"],
         color="darkblue", lw=1.5)
ax2.set_ylabel("Cum. Precip (mm)")
ax2.set_title("(c) Cumulative Precipitation", loc='left', fontweight='bold')
fmt_time_axis(ax2)

# ── Panel (d): Reservoir Water Level ──────────────────────────────────────────
vollstau = -0.595
mask_v  = ~df_res_daily["WL"].isna()
dates_v = df_res_daily["Zeit"][mask_v]
wl_v    = df_res_daily["WL"][mask_v]

ax3.plot(dates_v, wl_v, color="red", lw=1.0, zorder=1)
colors_wl = np.where(wl_v >= vollstau, "green", "blue")
ax3.scatter(dates_v, wl_v, color=colors_wl, s=8, edgecolor="none", zorder=2)
ax3.set_ylabel("WL (m)")
ax3.set_title(f"(d) Neuer Teich Water Level  [Maximum Capacity is at {vollstau} m]",
              loc='left', fontweight='bold')
wl_min, wl_max = wl_v.min(), wl_v.max()
ax3.set_yticks(np.arange(np.floor(wl_min), np.ceil(wl_max) + 1, 1.0))
fmt_time_axis(ax3)

# ── Panel (e): Artificial Reservoir Volume ────────────────────────────────────
# ── Panel (e): Artificial Reservoir Volume ────────────────────────────────────
ax4.fill_between(
    df_ard["Zeit"], 0, df_ard["ARR_m3"],
    where=df_ard["ARR_m3"] >= 0,
    interpolate=True,
    color="#1f77b4", alpha=0.75,
    label="Artificial Reservoir Volume"
)
ax4.plot(df_ard["Zeit"], df_ard["ARR_m3"],
         color="#0c5490", lw=0.8, zorder=3)
ax4.axhline(0, color="black", lw=0.8, ls="--", alpha=0.6)

ax4.set_xlabel("Year")
ax4.set_ylabel("Volume (m³)")
ax4.set_title("(e) Artificial Reservoir Volume", loc='left', fontweight='bold')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax4.legend(loc="upper left", fontsize=7)
fmt_time_axis(ax4)
# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig("double_mass_figure.png", bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
#DAHITI Data of Two Reservoir in side the catchment

In [ ]:

file_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\DAHITI Water Level data.xlsx"

df_alter = pd.read_excel(file_path, sheet_name="Alter_Teich")
df_mittel = pd.read_excel(file_path, sheet_name="Mittel_Teich")
df_Teich_3 = pd.read_excel(file_path, sheet_name="Teich_3")

df_alter["Date"] = pd.to_datetime(df_alter["Date"], dayfirst=True)
df_mittel["Date"] = pd.to_datetime(df_mittel["Date"], dayfirst=True)
df_Teich_3["Date"] = pd.to_datetime(df_Teich_3["Date"], dayfirst=True)
# Plotting
fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)  # share X-axis

# Panel 1: Alter Teich

axes[0].plot(df_alter["Date"], df_alter["WL"], color="blue", marker='o', lw=1.5, label="Alter Teich")
axes[0].set_ylabel("Water Level [m]")
axes[0].set_title("Alter Teich Water Level")
axes[0].grid(True, alpha=0.5)
axes[0].legend()

# Y-axis scaling 0.5 m
wl_min_1 = df_alter["WL"].min()
wl_max_1 = df_alter["WL"].max()
axes[0].set_yticks(np.arange(np.floor(wl_min_1*2)/2, np.ceil(wl_max_1*2)/2 + 0.5, 0.5))


# Panel 2: Mittel Teich

axes[1].plot(df_mittel["Date"], df_mittel["WL"], color="green", marker='s', lw=1.5, label="Mittel Teich")
axes[1].set_ylabel("Water Level [m]")
axes[1].set_title("Mittel Teich Water Level")
axes[1].grid(True, alpha=0.5)
axes[1].legend()

wl_min_2 = df_mittel["WL"].min()
wl_max_2 = df_mittel["WL"].max()
axes[1].set_yticks(np.arange(np.floor(wl_min_2*2)/2, np.ceil(wl_max_2*2)/2 + 0.5, 0.5))

# Panel 3: Teich_3

axes[2].plot(df_Teich_3["Date"], df_Teich_3["WL"], color="cyan", marker='o', lw=1.5, label="Teich_3")
axes[2].set_ylabel("Water Level [m]")
axes[2].set_title("Teich_3 Water Level")
axes[2].grid(True, alpha=0.5)
axes[2].legend()

# Y-axis scaling 0.5 m
wl_min_3 = df_Teich_3["WL"].min()
wl_max_3 = df_Teich_3["WL"].max()
axes[2].set_yticks(np.arange(np.floor(wl_min_3*2)/2, np.ceil(wl_max_3*2)/2 + 0.5, 0.5))

# X-axis formatting (shared)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%d.%m.%Y"))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# --------------------------------------------------
# Read hourly rainfall
# --------------------------------------------------
precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"

df_p = pd.read_csv(precip_path, sep=';')
df_p.columns = df_p.columns.str.strip()

df_p["Datum"] = pd.to_datetime(df_p["datetime"], dayfirst=True, errors="coerce")
df_p = df_p.dropna(subset=["Datum"])

# --------------------------------------------------
# Hourly → daily sum
# --------------------------------------------------
df_p_daily = (
    df_p
    .set_index("Datum")
    .resample("D")["precip"]
    .sum()
    .reset_index()
)

df_p_daily["precip"] = df_p_daily["precip"].clip(lower=0)

# --------------------------------------------------
# Time helpers (MATCHING YOUR DESIGN)
# --------------------------------------------------
df_p_daily["Jahr"] = df_p_daily["Datum"].dt.year

# Fix date to year 2000 for plotting (Jan–Dec alignment)
df_p_daily["Monatstag"] = pd.to_datetime(
    "2000-" + df_p_daily["Datum"].dt.strftime("%m-%d"),
    format="%Y-%m-%d"
)

# --------------------------------------------------
# Year-wise cumulative rainfall
# --------------------------------------------------
df_p_daily["Kumulierter_Niederschlag"] = (
    df_p_daily
    .groupby("Jahr")["precip"]
    .cumsum()
)


In [ ]:
def plot_jahre_niederschlag(
    df,
    ylabel,
    title,
    ylim=None,
    leg_pos="upper left",
    minor_interval=50,
    major_interval=200,
    padding=45,
    output_file=None
):
    plt.figure(figsize=(12, 7))

    letzter_tag = df["Datum"].max().strftime("%d.%m.%Y")

    for i, jahr in enumerate(jahre_alle):
        gruppe = df[df["Jahr"] == jahr]
        if gruppe.empty:
            continue

        plt.plot(
            gruppe["Monatstag"],
            gruppe["Kumulierter_Niederschlag"],
            label=str(jahr),
            color=farben(i),
            linewidth=2
        )

    # X-axis (IDENTICAL to your plot_jahre)
    plt.xticks(alle_ticks, alle_labels, rotation=45, fontsize=14)
    plt.xlim(pd.to_datetime("2000-01-01"), pd.to_datetime("2000-12-31"))

    # Y-axis
    if ylim is not None:
        plt.ylim(ylim)

    plt.ylabel(ylabel, fontsize=16, labelpad=padding)
    plt.title(title, fontsize=18)
    plt.legend(loc=leg_pos, fontsize=12, ncol=2)
    plt.grid(True, linestyle=":")

    # Footer text
    plt.gcf().text(
        0.975, 0.13, f"Stand: {letzter_tag}",
        ha="right", va="bottom", fontsize=14,
        bbox=dict(facecolor="white", edgecolor="grey", alpha=0.9,
                  boxstyle="round,pad=0.3")
    )

    plt.tight_layout()

    if output_file:
        plt.savefig(output_file, dpi=300)
        print(f"Plot gespeichert als: {output_file}")

    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd

# --- Example dataframe: df_p_daily with columns 'Jahr', 'Monatstag', 'Kumulierter_Niederschlag' ---
# df_p_daily['Monatstag'] should be datetime objects fixed to a dummy year (e.g., 2000)

jahre_alle = list(range(2015, 2024))
farben = cm.get_cmap('tab20', len(jahre_alle))

plt.figure(figsize=(12, 7))

# Plot each year's cumulative rainfall
for i, jahr in enumerate(jahre_alle):
    gruppe = df_p_daily[df_p_daily["Jahr"] == jahr]
    if gruppe.empty:
        continue
    plt.plot(
        gruppe["Monatstag"],
        gruppe["Kumulierter_Niederschlag"],
        label=str(jahr),
        color=farben(i),
        linewidth=2.5
    )

# --- X-axis: months ---
plt.xticks(
    pd.to_datetime([f'2000-{str(m).zfill(2)}-01' for m in range(1, 13)]),
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
    rotation=45, fontsize=14
)
plt.xlim(pd.to_datetime("2000-01-01"), pd.to_datetime("2000-12-31"))

# --- Y-axis with minor/major ticks ---
y_min, y_max = 0, 1400
minor_interval = 50
major_interval = 200
plt.ylim(y_min, y_max)

# Major ticks
yticks_major = np.arange(y_min, y_max + major_interval, major_interval)
plt.yticks(yticks_major, labels=[str(int(y)) for y in yticks_major], fontsize=14)

# Minor ticks (exclude major ticks)
yticks_minor = np.arange(y_min, y_max + minor_interval, minor_interval)
yticks_minor_filtered = [y for y in yticks_minor if y not in yticks_major]

# Draw minor grid lines + small labels
for y in yticks_minor_filtered:
    plt.axhline(y=y, color='gray', linestyle=':', linewidth=0.5, zorder=1)
    plt.text(pd.to_datetime('1999-12-27'), y, str(int(y)), va='center', ha='right', fontsize=7, zorder=2)


# --- Labels, title, legend ---
plt.ylabel("Cummulative Rainfall (mm)", fontsize=16, labelpad=45)
plt.title("Cummulative sum of Yearly Rainfall_Ziegenrück", fontsize=18)
plt.grid(True, linestyle=':')
plt.legend(loc='upper left', fontsize=12, ncol=2)
plt.tight_layout()
plt.show()
